# 02 - Groq com LangChain

Usando o Groq dentro do LangChain com `ChatGroq`, você tem o melhor dos dois mundos:
a **velocidade do Groq** com toda a infraestrutura do LangChain (chains, agentes, RAG, memória).

A interface é idêntica ao `ChatOpenAI` — trocar de provedor é questão de mudar o import.

| Provedor | Import | Variável de ambiente |
|----------|--------|---------------------|
| OpenAI | `from langchain_openai import ChatOpenAI` | `OPENAI_API_KEY` |
| Groq | `from langchain_groq import ChatGroq` | `GROQ_API_KEY` |
| Google | `from langchain_google_genai import ChatGoogleGenerativeAI` | `GEMINI_API_KEY` |
| Mistral | `from langchain_mistralai import ChatMistralAI` | `MISTRAL_API_KEY` |

### Pacotes necessários

```bash
pip install langchain-groq python-dotenv
```

In [1]:
from dotenv import find_dotenv, load_dotenv
from langchain_groq import ChatGroq

load_dotenv(find_dotenv())

# temperature=0: respostas determinísticas — ideal para tarefas factuais
chat = ChatGroq(
    temperature=0,
    model="llama-3.3-70b-versatile"  # atualizado do llama3-8b-8192
)

response = chat.invoke("Oi! Responda em português!")
print(response.content)

Olá! Claro, posso responder em português. Como posso ajudar você hoje? Você tem alguma pergunta ou precisa de ajuda com algo específico? Estou aqui para ajudar!


## 1. Usando com ChatPromptTemplate e LCEL

O `ChatGroq` é totalmente compatível com o operador `|` do LCEL.
Basta substituir o `ChatOpenAI` pelo `ChatGroq` — o restante da chain permanece igual.

In [2]:
import textwrap
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def formatar_texto(texto: str, largura: int = 100) -> None:
    print(textwrap.fill(texto, width=largura))

template = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente que sempre fala no sentido figurado"),
    ("human", "{input}")
])

chain = template | chat | StrOutputParser()

response = chain.invoke({"input": "O que é o céu?"})
formatar_texto(response)

O céu é um manto de azul infinito que envolve nosso mundo, um véu de mistério que nos separa do
desconhecido. É um espelho que reflete as nossas emoções, mudando de cor e textura de acordo com o
nosso estado de espírito. Quando estamos felizes, o céu é um brilhante azul cristalino, cheio de
promessas e possibilidades. Mas quando estamos tristes, ele se torna um cinza sombrio, um lembrete
de que a vida é cheia de altos e baixos.  O céu também é um palco para os sonhos, um lugar onde as
nuvens se transformam em criaturas mitológicas e as estrelas se tornam diamantes cintilantes. É um
lugar onde a imaginação não tem limites, onde podemos voar com as águias e dançar com as estrelas.
Mas, acima de tudo, o céu é um lembrete de que somos pequenos, mas fazemos parte de algo muito maior
do que nós mesmos. É um convite para olhar para cima, para além do nosso pequeno mundo, e para nos
conectar com o universo infinito que nos cerca.


## 2. Streaming com LCEL

O `.stream()` do LCEL funciona nativamente com o `ChatGroq`.
Cada chunk chega assim que é gerado pelo modelo.

In [3]:
stream = chain.stream({"input": "O que é viver?"})

for chunk in stream:
    print(chunk, end="", flush=True)

Viver é como navegar em um oceano sem fim, onde as ondas da vida nos levam a lugares desconhecidos e inesperados. É um tapete mágico que nos transporta para territórios de alegria e tristeza, de luz e sombra. É um jardim que precisamos cultivar com cuidado e dedicação, regando as flores da esperança e colhendo os frutos da experiência.

Viver é como uma obra de arte em constante evolução, onde cada dia é uma nova pincelada na tela da existência. É um concerto de sons e silêncios, onde a música da vida nos faz dançar ao ritmo do destino. É um labirinto de escolhas e decisões, onde cada caminho nos leva a novas descobertas e surpresas.

Viver é, acima de tudo, um presente precioso que devemos abrir com cuidado e apreciar com gratidão. É um fio de seda que nos liga ao universo e às pessoas ao nosso redor, e que nos permite tecer uma tapeçaria de amor, compaixão e conexão. É um sonho que nos permite voar alto e alcançar as estrelas, e que nos faz sentir vivos, pulsantes e cheios de vida.

## 3. Tools com ChatGroq

O `ChatGroq` suporta `bind_tools()` da mesma forma que o `ChatOpenAI`.
O Groq executa o raciocínio de tool calling no hardware LPU — muito rápido.

> **`tool_choice="auto"`**: o modelo decide se precisa ou não chamar a tool.
> Use `tool_choice="required"` para forçar sempre uma chamada.

In [4]:
from typing import Optional
from langchain_core.tools import tool
from datetime import datetime

@tool
def hora_atual(formato: Optional[str] = "%H:%M:%S") -> str:
    """Retorna a hora atual no formato especificado."""
    return datetime.now().strftime(formato)

# Vinculando a tool ao modelo
chat_com_tool = chat.bind_tools([hora_atual], tool_choice="auto")

response = chat_com_tool.invoke("Qual é a hora agora?")

print(f"Tool calls: {response.tool_calls}")
print(f"\nNome da tool: {response.tool_calls[0]['name']}")
print(f"Argumentos:   {response.tool_calls[0]['args']}")
print(f"ID da call:   {response.tool_calls[0]['id']}")

Tool calls: [{'name': 'hora_atual', 'args': {'formato': '%H:%M:%S'}, 'id': 'p2hzpqn84', 'type': 'tool_call'}]

Nome da tool: hora_atual
Argumentos:   {'formato': '%H:%M:%S'}
ID da call:   p2hzpqn84


## Resumo

| Operação | Código |
|----------|--------|
| Criar modelo | `ChatGroq(model="llama-3.3-70b-versatile", temperature=0)` |
| Usar em chain LCEL | `template \| chat \| StrOutputParser()` |
| Streaming | `chain.stream({"input": ...})` |
| Vincular tools | `chat.bind_tools([minha_tool])` |

> **Vantagem do Groq**: latência muito baixa — respostas em frações de segundo.
> Ideal para aplicações interativas onde a velocidade de resposta é crítica.